# M7: replication report (PLAN §7 M7, §10; D-102)

**Notebook settings:** Accelerator **None (CPU)** · Internet **on** · Secrets: **HF_TOKEN**. Run as **Save Version → Save & Run All**. Expected ≈ 20–30 min (most of it the install).

1. Restores this study's `results/<run>/` folders (the M4–M6 aggregates, `results/m6/` and the NF4 run) from the private store into the checkout.
2. **R6 timing** (T3 analogue): restores the stage logs into ephemeral `/tmp/rbbd_m7_cache` (`manifests`, `embed`, `deltab`, `generate`, and only the `*.json` sidecars of `embeddings/<slug>`), then `collect-timing` writes `results/<run>/timing.json` (numbers only).
3. `report` writes `results/REPORT.md`, `results/report/*.csv` and `results/figures/*.png`; a second call must log `cache hit: report`; the leakage test (DC-18) checks `results/`.
4. Syncs `results/` to the private store and packs `/kaggle/working/results_m7.tar.gz` (aggregates only) for download.

In [ ]:
REPO = "SGupta-4/Reference-Based-Bias-Detection-Implementation"
REF = "claude/kaggle-bias-detection-plan-h9es1w"  # a branch, tag or FULL 40-char SHA
WORK = "/kaggle/working/rbbd"
CONFIGS = ["tier2_qwen2.5-0.5b", "tier2_llama3.2-1b", "tier1_llama3.1-8b", "tier1_mistral-7b", "tier1_gemma3-4b"]
EPHEMERAL = "/tmp/rbbd_m7_cache"   # stage logs + embedding sidecars; lost at session end, nothing deleted
import os, time
SESSION_T0 = time.time()

In [ ]:
# Clone the repo at REF (branch, tag or commit SHA). If the repo is private, attach the
# GITHUB_TOKEN secret: it is passed to git as an HTTP header through environment
# variables, never in the URL, argv or output.
import base64, datetime, os, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ.setdefault(
    "RBBD_SESSION_ID", datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
RBBD_SESSION_ID = os.environ["RBBD_SESSION_ID"]  # also expands $RBBD_SESSION_ID in ! lines
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # fail instead of prompting for a password
try:
    _gh = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    _gh = None
if _gh:
    _basic = base64.b64encode(f"x-access-token:{_gh}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {_basic}")
del _gh

shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK)
for args in (["init", "-q"], ["remote", "add", "origin", f"https://github.com/{REPO}.git"],
             ["fetch", "-q", "--depth", "1", "origin", REF], ["checkout", "-q", "FETCH_HEAD"]):
    done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode and args[0] == "fetch" and "GIT_CONFIG_COUNT" in git_env:
        # An expired or revoked token gets HTTP 401 even on a public repo (B-025): retry
        # without it. git's stderr never contains the header value.
        print("GITHUB_TOKEN was rejected (expired or revoked?); retrying without it")
        git_env = {k: v for k, v in git_env.items() if not k.startswith("GIT_CONFIG_")}
        done = subprocess.run(["git", *args], cwd=WORK, env=git_env, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(f"git {args[0]} failed: {done.stderr[-500:]}")
del git_env
os.chdir(WORK)
print("commit", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
      "| session", os.environ["RBBD_SESSION_ID"])

In [ ]:
# B-032 / D-087: the Kaggle image runs Python 3.13 and vllm 0.10.1.1 has no 3.13 wheels.
# The pinned stack goes into a Python 3.12 venv on ephemeral disk. Every uv call names its
# target interpreter explicitly, and the image's own UV_* settings are cleared first: they
# made uv install into the system Python (B-032, second run). Putting the venv's bin/ first
# on PATH makes every `!python ...` line and the stage subprocesses use it. This kernel
# only imports rbbd's light modules (config, runner, manifests) from src/.
import os, shutil, subprocess

def sh(*cmd, tail=25):
    # Run a setup command; show the end of its output; stop the notebook if it fails.
    r = subprocess.run(list(cmd), capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().splitlines()
    print(f"$ {' '.join(cmd[:4])} ... -> exit {r.returncode}")
    if tail:
        print("\n".join(out[-tail:]))
    if r.returncode:
        raise RuntimeError(f"setup step failed: {' '.join(cmd[:4])}")
    return r.stdout

uv_vars = sorted(k for k in os.environ if k.startswith("UV_"))
print("image UV_* settings cleared:", uv_vars)  # names only
for k in uv_vars:
    del os.environ[k]
EPH = next((p for p in ("/kaggle/tmp", "/kaggle/temp") if os.path.isdir(p)), "/tmp")
VENV = f"{EPH}/rbbd_py312"
PY = f"{VENV}/bin/python"
os.environ.update(UV_CACHE_DIR=f"{EPH}/uv_cache", UV_LINK_MODE="copy",
                  UV_PYTHON_INSTALL_DIR=f"{EPH}/uv_python")
sh("pip", "install", "-q", "uv==0.8.17")
UV = shutil.which("uv")
sh(UV, "python", "install", "3.12")
sh(UV, "venv", "--python-preference", "only-managed", "--python", "3.12", VENV)
sh(PY, "--version")
sh(UV, "pip", "install", "--python", PY, "-e", ".[train,bench,dev]")
os.environ["VIRTUAL_ENV"] = VENV
os.environ["PATH"] = f"{VENV}/bin:" + os.environ["PATH"]
assert shutil.which("python") == PY, shutil.which("python")
with open("/kaggle/working/pip_freeze_m7.txt", "w") as fh:
    fh.write(sh(UV, "pip", "freeze", "--python", PY, tail=0))
# Preflight (B-012, D-049): stop here, not inside a stage, if the pinned stack does not import.
print(sh(PY, "-c", "import sys, torch, torchvision, transformers, vllm; print('preflight ok', "
         "sys.version.split()[0], torch.__version__, torchvision.__version__, "
         "transformers.__version__, vllm.__version__)"))

In [ ]:
# Copy Kaggle Secrets into the environment (presence flags only are printed) and keep the
# HF cache on ephemeral disk so it never counts against /kaggle/working (D-029/D-041).
import sys
sys.path.insert(0, f"{WORK}/src")  # the editable install is not visible to this already-running kernel
from rbbd.utils.env import ephemeral_dir, load_kaggle_secrets

_secrets = load_kaggle_secrets(["HF_TOKEN"])
print(_secrets)
if not _secrets["HF_TOKEN"]:
    # Stop here instead of failing later on every gated download (B-022).
    raise RuntimeError("HF_TOKEN secret not attached: Add-ons -> Secrets -> enable HF_TOKEN "
                       "for this notebook, then run again")
os.environ["HF_HOME"] = str(ephemeral_dir() / "hf")
ENV_DIR = f"/kaggle/working/artifacts/env/{os.environ['RBBD_SESSION_ID']}"
os.makedirs(ENV_DIR, exist_ok=True)

In [ ]:
# Restore the aggregates, then collect R6 timing per config from the stage logs (ephemeral disk).
import json, shutil, subprocess
from pathlib import Path
from rbbd import config as _config
ROOT = Path("/kaggle/working/artifacts")
RES = Path(WORK) / "results"            # the checkout's results/ (what git and the leakage test see)
EPH_ENV = {**os.environ, "RBBD_ARTIFACTS": EPHEMERAL}
SUMMARY = {}
# Only this study's runs: the store also holds e.g. the smoke run, which shares a model slug.
RUNS = ["m6", "m6_nf4_llama3.1-8b"] + [_config.load(f"configs/{n}.yaml")["run_name"] for n in CONFIGS]
for RUN in RUNS:
    !python -m rbbd.cli restore --path results/{RUN}
    shutil.copytree(ROOT / "results" / RUN, RES / RUN, dirs_exist_ok=True)
assert (RES / "m6" / "t1.csv").exists(), "results/m6 missing in the store: run m6_analysis first"
for path in ["manifests", "embed", "deltab", "generate"]:
    !RBBD_ARTIFACTS={EPHEMERAL} python -m rbbd.cli restore --path {path}
for NAME in CONFIGS:
    cfg = _config.load(f"configs/{NAME}.yaml")
    SLUG = cfg["models"][0]["slug"]
    !RBBD_ARTIFACTS={EPHEMERAL} python -m rbbd.cli restore --path embeddings/{SLUG} --include "*.json"
    r = subprocess.run(["python", "-m", "rbbd.cli", "collect-timing", "--config", f"configs/{NAME}.yaml"],
                       capture_output=True, text=True, env=EPH_ENV)
    SUMMARY[NAME] = r.stdout.strip()[-600:] if r.returncode == 0 else f"FAILED: {r.stderr[-800:]}"
    print(NAME, SUMMARY[NAME])
!df -h /kaggle/working {EPHEMERAL} | tail -2

In [ ]:
# The report, the cache re-run (must hit) and the leakage test; then sync and pack.
t0 = time.time()
!python -m rbbd.cli report --results {RES} > /kaggle/working/m7_summary.json
SUMMARY["report_seconds"] = round(time.time() - t0, 1)
r = subprocess.run(["python", "-m", "rbbd.cli", "report", "--results", str(RES)], capture_output=True, text=True)
SUMMARY["report_cache_hit"] = "cache hit: report" in (r.stdout + r.stderr)
r = subprocess.run(["python", "-m", "pytest", "-q", "-p", "no:warnings", "tests/test_leakage.py"],
                   capture_output=True, text=True)
SUMMARY["dc18_leakage"] = (r.stdout.strip().splitlines() or ["?"])[-1]
if "failed" in SUMMARY["dc18_leakage"]:
    raise RuntimeError("leakage test failed; results/ not synced: " + r.stdout[-1500:])
shutil.copytree(RES, ROOT / "results", dirs_exist_ok=True)
!python -m rbbd.cli sync --path results
!tar -czf /kaggle/working/results_m7.tar.gz -C {WORK} results
print(json.dumps(SUMMARY, indent=1))

In [ ]:
# The verdict section and the cost table (aggregates only).
text = (RES / "REPORT.md").read_text()
start, end = text.index("## Verdict"), text.index("### Per setting")
print(text[start:end])
r6 = text.index("## R6"); print(text[r6:text.index("## Sensitivity")])
!python -m rbbd.cli sync --path env/$RBBD_SESSION_ID
print("session hours:", round((time.time() - SESSION_T0) / 3600, 2))